# Calculate all subjects' dissim ~ genmag, spread

20251203 ~ 20260102

To create the library, implement and test the function here.

Calculate genmag and spread then store in work/data/processed as hdf style.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

project_root = os.path.abspath("/home/jovyan/work")
print("Project root:", project_root)

if project_root not in sys.path:
    sys.path.append(project_root)

from funcs import core, viz

In [ ]:
d_path = "/home/jovyan/work/data/raw/Amy_dissimilarity.csv"
unique_words = ["red", "orange", "yellow", "green", "blue", "purple", "pink", "brown", "grey", "black", "happiness", "joy", "confidence", "calm", "boredom", "confusion", "anxiety", "fear", "sadness", "defeated", "anger", "envy", "disgust"]
n_sub = 120
raw = pd.read_csv(d_path, header=None)
raw.drop(columns=0, inplace=True)
raw

In [ ]:
# define t

t1 = np.logspace(-4, -1, 20, endpoint=False)   # 10^-4 ～ 10^-1
t2 = np.logspace(-1,  2, 500, endpoint=False)  # 10^-1 ～ 10^2
t3 = np.logspace(2,  4, 20, endpoint=False)    # 10^2 ～ 10^4
t4 = np.logspace(4, 8, 20, endpoint=True)

t_all = np.concatenate([t1, t2, t3, t4])

In [ ]:
genmag_all = pd.DataFrame(
    data=np.nan,
    index=np.arange(n_sub),
    columns=t_all,
    dtype=float
)

spr_all = pd.DataFrame(
    data=np.nan,
    index=np.arange(n_sub),
    columns=t_all,
    dtype=float
)

positive_definite_t_all = pd.DataFrame(
    data=pd.NA,
    index=np.arange(n_sub),
    columns=t_all,
    dtype="boolean"
)

In [ ]:
h5_path = project_root + "/data/processed/amy_processeddata_all.h5"

with pd.HDFStore(h5_path, mode="w") as store:

    for sub in range(n_sub):
        print(f'subject: {sub}')
        dissim_mtx = core.create_dissim_amy(raw, sub_no=sub, unique_words=unique_words)
        dist = core.cal_dissim2dist(dissim_mtx)
        store.put(f"/dissim/sub_{sub:03d}", dissim_mtx, format="fixed")

        for i_t, t in enumerate(t_all):
            sim = core.cal_dist2sim(dist, t)
            genmag = core.cal_sim2genmag(sim)
            genmag_all.iloc[sub, i_t] = genmag

            spr = core.cal_dist2spread(dist, t)
            spr_all.iloc[sub, i_t] = spr

            A = sim.to_numpy()
            is_pd = core.check_positivedefinite(A)
            positive_definite_t_all.iloc[sub, i_t] = bool(is_pd)

In [ ]:
if genmag_all.columns.is_unique is False or spr_all.columns.is_unique is False:
    raise ValueError("Keys of columns are not unique. Recommend checking t setting.")


positive_definite_df = positive_definite_t_all.astype("bool") 
with pd.HDFStore(h5_path, mode="a") as store:
    store.put("/metrics/genmag", genmag_all, format="fixed")
    store.put("/metrics/spread", spr_all, format="fixed")
    store.put("/metrics/positivedefinite", positive_definite_df, format="fixed")

In [ ]:
output = os.path.join(project_root, "fig/temp")

viz.plt_metric_individual(
    genmag_all,
    metric_name="Generalized magnitude",
    output_dir=output,
    nrows=5, ncols=4,
    ylim=[-1, 24], xlim=None,
    xticks_all=True, yticks_all=True,
    positive_definite_df=positive_definite_t_all,
    posdef_mode= "two_color",
)

In [ ]:
# all subjects mean
n_sub = 1
i_sub = 0

genmag_mean = pd.DataFrame(
    data=np.nan,
    index=np.arange(n_sub),
    columns=t_all,
    dtype=float
)

spr_mean = pd.DataFrame(
    data=np.nan,
    index=np.arange(n_sub),
    columns=t_all,
    dtype=float
)

positive_definite_t_mean = pd.DataFrame(
    data=pd.NA,
    index=np.arange(n_sub),
    columns=t_all,
    dtype="boolean"
)

dissim_mtx = core.create_dissim_amy(raw, sub_no=-1, unique_words=unique_words)
dist = core.cal_dissim2dist(dissim_mtx)

viz.plt_dissim_heatmap(
    dissim_mtx,
    mode_dissim=True,
    save=True,
    figpath=project_root + "/fig/dissim/dissim_mean.jpg")

In [ ]:
for i_t, t in enumerate(t_all):
    sim = core.cal_dist2sim(dist, t)
    genmag = core.cal_sim2genmag(sim)
    genmag_mean.iloc[0, i_t] = genmag

    spr = core.cal_dist2spread(dist, t)
    spr_mean.iloc[0, i_t] = spr

    A = sim.to_numpy()
    is_pd = core.check_positivedefinite(A)
    positive_definite_t_mean.iloc[i_sub, i_t] = bool(is_pd)

In [ ]:
h5_path_mean = project_root + "/data/processed/amy_processeddata_mean.h5"

positive_definite_df_mean = positive_definite_t_mean.astype("bool")
with pd.HDFStore(h5_path_mean, mode="w") as store:
    store.put("/dissim/mean", dissim_mtx, format="fixed")
    store.put("/metrics/genmag", genmag_mean, format="fixed")
    store.put("/metrics/spread", spr_mean, format="fixed")
    store.put("/metrics/positivedefinite", positive_definite_df_mean, format="fixed")